<a href="https://colab.research.google.com/github/asfiya-tehmeen/Clinical-Trial-Patient-Matching-Agent/blob/main/criteria_reasoning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CELL 9 — Criteria reasoning agent (the core — uses grok-3)

# For each trial, Grok checks every inclusion/exclusion criterion.
# Each criterion gets: PASS / FAIL / UNCERTAIN / MISSING_DATA

CRITERIA_PROMPT = """
You are a clinical trial eligibility specialist with deep knowledge of
oncology trial protocols.

PATIENT PROFILE:
{patient_json}

CLINICAL TRIAL: {nct_id}
Title: {title}

ELIGIBILITY CRITERIA:
{eligibility_text}

For each inclusion and exclusion criterion, determine if this patient meets it.
Return ONLY valid JSON — no preamble, no markdown fences.

{{
  "trial_nct_id": "{nct_id}",
  "overall_eligible": true | false | "uncertain",
  "criteria_results": [
    {{
      "criterion_text": "<exact criterion text, shortened to 80 chars>",
      "type": "inclusion" | "exclusion",
      "verdict": "PASS" | "FAIL" | "UNCERTAIN" | "MISSING_DATA",
      "confidence": <float 0.0-1.0>,
      "reasoning": "<one sentence citing specific patient data>",
      "missing_info": "<if MISSING_DATA: what test or info resolves this, else null>"
    }}
  ],
  "disqualifying_criteria": ["<criterion text if FAIL>"],
  "uncertain_criteria": ["<criterion text if UNCERTAIN>"],
  "missing_data_requests": ["<actionable test or data needed, if any>"],
  "match_summary": "<2 sentence plain-English summary for a physician>"
}}

Rules:
- PASS         = patient clearly meets this criterion based on available data
- FAIL         = patient clearly does NOT meet this criterion (disqualifying)
- UNCERTAIN    = ambiguous — not enough info to decide either way
- MISSING_DATA = patient may qualify but a specific piece of data is absent
- If ANY inclusion criterion is FAIL → overall_eligible = false
- If ANY exclusion criterion is PASS (patient HAS the exclusion) → overall_eligible = false
- Be specific — cite actual values (e.g. "ECOG 1 meets PS 0-2 requirement")
"""

def reason_over_trial(trial: dict, patient_profile: dict) -> dict:
    """Run Grok-3 reasoning over a single trial's eligibility criteria."""
    prompt = CRITERIA_PROMPT.format(
        patient_json     = json.dumps(patient_profile, indent=2),
        nct_id           = trial["nct_id"],
        title            = trial["title"],
        eligibility_text = trial["eligibility_text"][:2500]
    )
    try:
        raw = call_grok(prompt, "llama-3.3-70b-versatile", max_tokens=2000)
        result = clean_json(raw)
        result["trial_title"] = trial["title"]
        result["trial_phase"] = trial["phase"]
        result["trial_url"]   = trial["url"]
        result["trial_sites"] = trial["sites"]
        result["sponsor"]     = trial["sponsor"]
        return result
    except Exception as e:
        return {
            "trial_nct_id"         : trial["nct_id"],
            "trial_title"          : trial["title"],
            "overall_eligible"     : False,
            "criteria_results"     : [],
            "match_summary"        : f"Parsing error: {e}",
            "missing_data_requests": [],
            "error"                : str(e)
        }

print(f" Running grok-3 reasoning on {len(pre_filtered)} trials...")
print("   (~5–10 sec per trial — grab a coffee)\n")

all_results = []
for i, trial in enumerate(pre_filtered):
    print(f"   [{i+1}/{len(pre_filtered)}] {trial['nct_id']} — {trial['title'][:55]}...")
    result = reason_over_trial(trial, patient_profile)
    all_results.append(result)
    time.sleep(0.5)

print(f"\n✅ Reasoning complete on {len(all_results)} trials")

 Running grok-3 reasoning on 24 trials...
   (~5–10 sec per trial — grab a coffee)

   [1/24] NCT06734182 — Neoadjuvant Envafolimab Plus Disitamab Vedotin and Carb...
   [2/24] NCT04267848 — Testing the Addition of a Type of Drug Called Immunothe...
   [3/24] NCT05061550 — Neoadjuvant and Adjuvant Treatment in Resectable Non-sm...
   [4/24] NCT05341583 — Ensartinib as Adjuvant Treatment in Anaplastic Lymphoma...
   [5/24] NCT05989542 — A Confirmatory Clinical Study in NSCLC Patients With ME...
   [6/24] NCT06269211 — Neoadjuvant Toripalimab for Clinically Stage II-IIIB Re...
   [7/24] NCT07534033 — Phase II Clinical Study of Probiotic LR607 in Patients ...
   [8/24] NCT07288034 — Immunotherapy Biomarkers to Predict First-line PD(L)1-b...
   [9/24] NCT06951464 — A Study of BL-B01D1 and Almonertinib in Patients With R...
   [10/24] NCT07003490 — Radical Resection With Contralateral Lymph Node Dissect...
   [11/24] NCT06788912 — Pembrolizumab (MK-3475) Plus Investigational Agents in ...
 